In [ ]:
# =========================
# 1. Import libraries
# =========================
import pandas as pd
import re
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report
from sklearn.impute import SimpleImputer


# =========================
# 2. Load dataset
# =========================
file_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\02. data\Full data.csv"
df = pd.read_csv(file_path)

# Fill missing values initially
df.fillna(0, inplace=True)


# =========================
# 3. Data cleaning function
# =========================
def clean_value(x):
    # Clean numeric values by removing non-numeric characters
    # and converting them to float. If conversion fails, return 0.
    if pd.isna(x):
        return 0
    cleaned_value = re.sub(r'[^\d.-]', '', str(x))
    try:
        return float(cleaned_value)
    except ValueError:
        return 0


# =========================
# 4. Feature preprocessing
# =========================
feature_columns = df.columns[2:70]

# Apply cleaning function to feature columns
for col in feature_columns:df[col] = df[col].apply(clean_value)

# Handle infinite and missing values
df[feature_columns] = df[feature_columns].replace([np.inf, -np.inf], 0)
df[feature_columns] = df[feature_columns].fillna(0)


# =========================
# 5. Create multi-label target variables
# =========================
df['Pictogram(s)'] = df['Pictogram(s)'].apply(lambda x: str(x).lower() if pd.notna(x) else '')
outputs = [
    'Irritant',
    'Acute Toxic',
    'Flammable',
    'Corrosive',
    'Health Hazard',
    'Environmental Hazard',
    'Compressed Gas',
    'Explosive',
    'Oxidizer'
]

# Convert text labels into binary format
for output in outputs:df[output] = df['Pictogram(s)'].apply(lambda x: 1 if output.lower() in x.lower() else 0)


# =========================
# 6. Train-test split
# =========================
X = df[feature_columns].astype(float)
y = df[outputs]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)


# =========================
# 7. Handle missing and infinite values
# =========================
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)
X_train = np.where(np.isinf(X_train), 0, X_train)
X_test = np.where(np.isinf(X_test), 0, X_test)


# =========================
# 8. Feature scaling
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================
# 9. Train Random Forest model
# =========================
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)


# =========================
# 10. Global feature importance
# =========================
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': X.columns,'Importance': importances}).sort_values(by='Importance', ascending=False)


# =========================
# 11. Feature importance per output class
# =========================
for output in outputs:
    print(f"Feature Importance for {output}:")
    y_train_output = y_train[output]
    rf_output = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_output.fit(X_train, y_train_output)
    output_importances = rf_output.feature_importances_
    output_importance_df = pd.DataFrame({'Feature': X.columns,'Importance': output_importances}).sort_values(by='Importance', ascending=False)
    print(output_importance_df)
    print("\n")


# =========================
# 12. Extract top 5 features per output
# =========================
top5_results = []

for output in outputs:
    y_train_output = y_train[output]
    rf_output = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_output.fit(X_train, y_train_output)
    output_importances = rf_output.feature_importances_
    output_importance_df = pd.DataFrame({'Feature': X.columns,'Importance': output_importances}).sort_values(by='Importance', ascending=False)
    top5 = output_importance_df.head(5).copy()
    top5['Output'] = outpu
    top5_results.append(top5)


# Combine results across outputs
top5_all_outputs = pd.concat(top5_results, ignore_index=True)
top5_all_outputs = top5_all_outputs[['Output', 'Feature', 'Importance']]


# =========================
# 13. Export results
# =========================
export_path = r"C:\Users\user\Desktop\u\Y3S1\ZCT 398\03. features selection\top5_feature_importances.csv"
top5_all_outputs.to_csv(export_path, index=False)
print("Exported feature importance results successfully!")